# Task 2 - A Transformer that basecalls nanopore signals

Read the Task 2 background in the assignment PDF before starting.

## 0. Setup

In [ ]:
import time, numpy as np, torch
from torch import nn
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASES = "ACGT"
print("device:", DEVICE)

## 1. Look at the data

The generator below is the ground truth for the whole task: it turns a random strand into a
squiggle. `noise` is the main data setting and `uneven` is the other one, both used in Section 4.

In [ ]:
#@title The data generator - run it, nothing to change { display-mode: "form" }
K, DWELL, N_BASES = 3, 8, 60
NOISE = 0.20                              # <-- Section 4 overrides this per run with noise=; leave it alone
NUM_KMERS = 4 ** K
READ_LEN = N_BASES * DWELL                # one read: 60 bases x 8 samples = 480 numbers
COMP = np.array([3, 2, 1, 0])             # complement: A<->T, C<->G

rng = np.random.default_rng(42)
KMER_LEVEL = rng.normal(0, 1, NUM_KMERS).astype(np.float32)   # the pore model

def kmer_ids(seqs):
    B, n = seqs.shape
    p = np.pad(seqs, ((0, 0), (K - 1, 0)), mode="wrap")
    ids = np.zeros((B, n), dtype=np.int64)
    for j in range(K):
        ids = ids * 4 + p[:, j:j + n]
    return ids

def make_batch(batch, noise=None, gen=None, return_adapter=False, uneven=0.0):
    '''signal (B, READ_LEN), label (B, N_BASES), fwd (B,) 1=forward 0=reverse, [adapter idx].

    A read is one strand of DNA pulled through the pore. Half the time it is the
    forward strand, half the time it is that strand's reverse complement. The
    current always comes from whichever strand is actually in the pore. The answer
    we want is always the forward strand.

    Reverse reads carry a leftover sequencing adapter, which the pore sees before any
    of the DNA and which shows up as a spike on the first base's samples.

    Two data settings:
      noise   - Gaussian noise strength on the current
      uneven  - if >0, the noise level VARIES per base (in [1-u, 1+u] * noise)
    '''
    g = gen or rng; noise = NOISE if noise is None else noise
    fwd_strand = g.integers(0, 4, size=(batch, N_BASES))      # the strand we must report
    fwd = g.integers(0, 2, size=batch)                         # 1 = forward, 0 = reverse
    # whichever strand is physically in the pore is what sets the current
    in_pore = np.where(fwd[:, None] == 1, fwd_strand, COMP[fwd_strand][:, ::-1])
    lvl = KMER_LEVEL[kmer_ids(in_pore)]                        # (B, N_BASES) level per base
    full = (np.arange(N_BASES + 1) * DWELL)[None, :].repeat(batch, 0)
    t = np.arange(READ_LEN)
    base_idx = np.clip((full[:, 1:, None] <= t[None, None, :]).sum(1), 0, N_BASES - 1)
    sig = np.take_along_axis(lvl, base_idx, axis=1).astype(np.float32)     # (B, READ_LEN)
    adapter = np.where(fwd == 0, 0, -1)         # the adapter, at the read start
    sig += 6.0 * ((base_idx == adapter[:, None]) & (fwd[:, None] == 0))     # adapter spike
    if uneven > 0:
        bscale = 1 + uneven * (2 * g.random((batch, N_BASES)) - 1)
        psc = np.take_along_axis(bscale, base_idx, axis=1)
        sig += (g.normal(0, 1, sig.shape) * noise * psc).astype(np.float32)
    else:
        sig += g.normal(0, noise, size=sig.shape).astype(np.float32)
    label = fwd_strand                  # always report the forward strand
    out = [torch.from_numpy(sig), torch.from_numpy(label.copy()), torch.from_numpy(fwd)]
    if return_adapter: out.append(torch.from_numpy(adapter))
    return tuple(out)

### A forward read and a reverse read

Run this a few times. What separates a forward read from a reverse one?

In [ ]:
#@title Plot a forward read and a reverse read { display-mode: "form" }
fig, axes = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
for ax, want in zip(axes, [1, 0]):
    while True:
        sig, lab, fwd = make_batch(1)
        if fwd[0] == want: break
    ax.plot(sig[0].numpy(), lw=.8)
    for i in range(N_BASES): ax.axvline(i * DWELL, color="k", ls=":", alpha=.15)
    ax.set_title("FORWARD" if want else "REVERSE")
    ax.set_ylabel("current")
axes[-1].set_xlabel("signal sample"); plt.tight_layout(); plt.show()

### The same read, with both candidate answers

The pore reads whichever strand it captured, but the answer is always the forward strand.
The cell below draws one reverse capture with both strings underneath it, so you can see
which part of the signal fixes which base.

In [ ]:
#@title One reverse read, with both candidate answers { display-mode: "form" }
# The same bookkeeping, on one real reverse read. Each coloured pair is one answer base
# and the part of the signal that actually determines it.
while True:
    sig, lab, fwd = make_batch(1)
    if fwd[0] == 0: break

answer  = "".join(BASES[c] for c in lab[0].tolist())
in_pore = answer[::-1].translate(str.maketrans("ACGT", "TGCA"))     # the reverse complement

fig, ax = plt.subplots(2, 1, figsize=(11, 4.6), gridspec_kw={"height_ratios": [3, 1.5]})
ax[0].plot(sig[0].numpy(), lw=.8)
ax[0].set_ylabel("current"); ax[0].set_title("one REVERSE read")
for a in ax: a.set_xlim(-70, READ_LEN)
ax[1].set_ylim(0, 1); ax[1].axis("off")

top, bot = [], []                       # keep the letter handles so we can recolour a few
for i in range(N_BASES):
    x = (i + .5) * DWELL
    top.append(ax[1].text(x, .72, in_pore[i], ha="center", va="center",
                          fontsize=7, family="monospace"))
    bot.append(ax[1].text(x, .18, answer[i], ha="center", va="center",
                          fontsize=7, family="monospace"))
ax[1].text(-8, .72, "in the pore", ha="right", va="center", fontsize=8)
ax[1].text(-8, .18, "the answer",  ha="right", va="center", fontsize=8)

for i, colour, rad in [(3, "crimson", .18), (22, "seagreen", -.30), (48, "darkorange", .45)]:
    j = N_BASES - 1 - i                                  # where that base's current sits
    for t in (top[j], bot[i]):
        t.set_color(colour); t.set_weight("bold"); t.set_fontsize(9)
    ax[1].annotate("", xy=((i + .5) * DWELL, .28), xytext=((j + .5) * DWELL, .64),
                   arrowprops=dict(arrowstyle="->", lw=1.3, color=colour, alpha=.9,
                                   connectionstyle=f"arc3,rad={rad}"))
plt.tight_layout(); plt.show()

## 2. First attempt: the CNN you already know (given)

Before reaching for anything new, try the architecture from Task 1. It is written out below.

Task 1's CNN finished by flattening everything into a
single number for the whole sequence. This one never flattens: 480 samples become 60
positions and stay 60 positions, and the last layer turns each one into its own 4 scores.
That is the only change needed to go from one label per sequence to one label per position.

| | |
|---|---|
| `stem` | each ratchet step's 8 samples into one token of width `C`, giving `(batch, C, 60)`. `C` is the channel count - the number of filters, as in Task 1. |
| `body` | mixes each token with its neighbours. `Conv1d` with `padding=kernel//2` keeps the length at 60. |
| `head` | `Conv1d(C, 4, kernel_size=1)`, which is a `Linear` applied at each position separately, giving `(batch, 4, 60)`. |

Run it, train it, and read the two accuracy columns separately.

In [ ]:
#@title CNN
class CNNBasecaller(nn.Module):
    """The Task 1 architecture, adapted: one label per base instead of one per sequence."""
    def __init__(self, C=112, layers=3, kernel=5):
        # C is the number of channels the body carries: how many different patterns the
        # network can track at each position. Same idea as the filter count in Task 1.
        # C=112 puts this CNN at about 190k parameters; `train` prints the count.
        super().__init__()
        # one token per ratchet step, exactly the tokenizer trick
        self.stem = nn.Conv1d(1, C, kernel_size=DWELL, stride=DWELL)
        # the body: mix each token with its neighbours. padding=kernel//2 adds that many
        # zeros at each end, so a width-k window still fits at every one of the 60
        # positions and the output is 60 long again (holds for any odd kernel).
        body = []
        for _ in range(layers):
            body += [nn.Conv1d(C, C, kernel, padding=kernel // 2), nn.ReLU()]
        self.body = nn.Sequential(*body)
        # the head: a width-1 conv is a Linear applied at each position on its own
        self.head = nn.Conv1d(C, 4, kernel_size=1)

    def forward(self, sig):                    # sig (batch, READ_LEN)
        x = self.stem(sig[:, None, :])         # -> (batch, C, N_BASES)
        x = self.body(x)                       # -> (batch, C, N_BASES)
        x = self.head(x)                       # -> (batch, 4, N_BASES)
        return x.transpose(1, 2)               # -> (batch, N_BASES, 4)

### CNN Training

Task 1 paired `sigmoid` with `BCEWithLogitsLoss` for a yes/no answer. The four-way version
is `softmax` with `nn.CrossEntropyLoss`, and `train` below already uses it. Like
`BCEWithLogitsLoss` it applies the softmax internally, so your head stays plain with no
softmax inside it. The called base is the largest of the four scores (`.argmax(-1)`).

`evaluate` reports accuracy **split by orientation**. Watch the two columns, not just the
overall number - the gap between them is the whole story of this task.

In [ ]:
#@title evaluate() and train() - run it, nothing to change { display-mode: "form" }
def evaluate(model, n=4000, seed=1234, **data):
    # fixed generator: the same model always scores the same, on the same reads
    model.eval()
    with torch.no_grad():
        sig, lab, fwd = make_batch(n, gen=np.random.default_rng(seed), **data)
        pred = model(sig.to(DEVICE)).argmax(-1).cpu()
    ok = (pred == lab).float().mean(1)
    return (pred == lab).float().mean().item(), ok[fwd == 1].mean().item(), ok[fwd == 0].mean().item()

def train(model, name="model", steps=3000, batch=128, lr=1.5e-3, seed=0, every=500, **data):
    """Train a model on freshly generated reads and report what happened.

    model   the network to train, already built
    name    label for the printed summary line
    steps   optimiser steps
    batch   reads per step
    lr      learning rate
    seed    seeds the batch order, NOT the weights - the model already exists by the time
            we get here. To fix the weights too, seed before you build it:
                torch.manual_seed(0); model = Basecaller(...)
    every   evaluate and print a progress line this often
    **data  forwarded to make_batch: noise=, uneven=

    Returns a dict:
        acc, fwd, rev   final accuracy overall and per orientation
        params          trainable parameter count
        hist            one entry per progress line, each holding step, loss, acc, fwd,
                        rev - this is what you plot to get a learning curve
    """
    torch.manual_seed(seed); model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr); lossf = nn.CrossEntropyLoss()
    gen = np.random.default_rng(7); t0 = time.time()
    hist, run = [], torch.zeros((), device=DEVICE)
    for s in range(1, steps + 1):
        sig, lab, _ = make_batch(batch, gen=gen, **data)
        loss = lossf(model(sig.to(DEVICE)).reshape(-1, 4), lab.reshape(-1).to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
        run += loss.detach()                       # kept on the GPU, no sync every step
        if s % every == 0:                         # progress point: record and print
            a_, f_, r_ = evaluate(model, **data)
            hist.append(dict(step=s, loss=(run / every).item(), acc=a_, fwd=f_, rev=r_))
            print(f"   step {s:>5}/{steps}   loss {hist[-1]['loss']:.3f}   "
                  f"acc {a_:.3f} (fwd {f_:.3f} | rev {r_:.3f})", flush=True)
            run = torch.zeros((), device=DEVICE); model.train()
    acc, af, ar = evaluate(model, **data)          # evaluate in the SAME regime it trained on
    print(f"{name:16s} params={sum(p.numel() for p in model.parameters()):>8,}  "
          f"acc={acc:.3f}  (forward {af:.3f} | reverse {ar:.3f})  {time.time()-t0:.0f}s")
    return dict(name=name, acc=acc, fwd=af, rev=ar, hist=hist,
                params=sum(p.numel() for p in model.parameters()))

In [ ]:
cnn = CNNBasecaller()
cnn_run = train(cnn, "CNN baseline")

## 3. Attention is all you need (TODO-1)

Self-attention computes each position's output as a weighted sum over every position in the
read. The weights are not fixed: they are computed from the content of the positions
themselves, so each position decides for itself where to look.

The five blocks below are ready to use.

| block | what it does | how you create it |
|---|---|---|
| `Tokenizer` | each ratchet step's 8 samples into one token of width `d_model` | `Tokenizer(d_model)` |
| `PositionalEncoding` | adds a fixed position signature to each token | `PositionalEncoding(d_model)` |
| `MultiHeadSelfAttention` | every token looks at every token, `n_heads` times in parallel | inside `EncoderBlock`, not built by you |
| `FeedForward` | a small per-token network of inner width `d_ff` | inside `EncoderBlock`, not built by you |
| `EncoderBlock` | one attention plus one feed-forward, with the residual connections and normalisation that keep a deep stack trainable | `EncoderBlock(d_model, n_heads, d_ff)` |

The output head on top is a plain `nn.Linear(d_model, 4)`.

The four sizes you hand those blocks are the configuration you pick, and `TODO-2` is where
you pick it on evidence:

| setting | what it controls |
|---|---|
| `d_model` | the width of a token, and the reason it appears in nearly every line above: it is the width of whatever passes between the blocks, so all of them have to agree on it. |
| `n_blocks` | how many `EncoderBlock`s you stack, one feeding into the next. |
| `n_heads` | how many attention patterns each block computes side by side, the *attention heads* from Tutorial 4. They divide `d_model` between them, so `d_model` must be a multiple of `n_heads`. |
| `d_ff` | the width of the small per-token network inside each block. It is normally a small multiple of `d_model`. |

In [ ]:
#@title The five blocks
class Tokenizer(nn.Module):
    "Collapse each base's DWELL signal samples into one token vector (a strided conv)."
    def __init__(self, d_model):
        super().__init__()
        self.conv = nn.Conv1d(1, d_model, kernel_size=DWELL, stride=DWELL)
    def forward(self, sig):                 # sig (B, READ_LEN)
        return self.conv(sig[:, None, :]).transpose(1, 2)     # (B, N_BASES, d_model)

class PositionalEncoding(nn.Module):
    "Add a fixed sinusoidal position signature so attention can tell positions apart."
    def __init__(self, d_model, max_len=N_BASES):
        super().__init__()
        assert d_model % 2 == 0, f"d_model ({d_model}) must be even"
        pos = torch.arange(max_len)[:, None]
        div = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe)
    def forward(self, x):
        return x + self.pe[None, :x.size(1)]

class MultiHeadSelfAttention(nn.Module):
    "Every token looks at every token: softmax(Q Kᵀ / sqrt(dk)) V, from scratch."
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, f"d_model ({d_model}) must divide evenly by n_heads ({n_heads})"
        self.h, self.dk = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
    def forward(self, x, return_attn=False):
        B, T, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.h, self.dk).transpose(1, 2) for t in (q, k, v)]
        att = (q @ k.transpose(-2, -1) / self.dk ** 0.5).softmax(dim=-1)
        y = (att @ v).transpose(1, 2).reshape(B, T, d)
        return (self.out(y), att) if return_attn else self.out(y)

class FeedForward(nn.Module):
    "Per-token 2-layer MLP that mixes features (not positions)."
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)

class EncoderBlock(nn.Module):
    "One encoder: attention, then feed-forward - each wrapped in a residual + LayerNorm."
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # attention + residual connection
        x = x + self.ff(self.ln2(x))       # feed-forward + residual connection
        return x

**The model.** The four blocks stacked into one module, in the order the PDF draws them:
tokenize, add the positional encoding, run the encoder blocks, then the output head.

`forward` is the whole architecture in four lines.

In [ ]:
#@title Basecaller
class Basecaller(nn.Module):
    def __init__(self, d_model, n_heads, n_blocks, d_ff, use_pe=True):
        super().__init__()
        # kept so a saved model can be rebuilt without you retyping its sizes
        self.sizes = dict(d_model=d_model, n_heads=n_heads, n_blocks=n_blocks,
                          d_ff=d_ff, use_pe=use_pe)
        self.tokenizer = Tokenizer(d_model)
        # use_pe=False drops the positional encoding - that is the Section 5 ablation
        self.pos = PositionalEncoding(d_model) if use_pe else None
        self.blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff)
                                     for _ in range(n_blocks)])
        self.head = nn.Linear(d_model, 4)     # the output head

    def forward(self, sig):
        x = self.tokenizer(sig)                    # -> (batch, N_BASES, d_model)
        if self.pos is not None: x = self.pos(x)   # same shape, positions added on
        for block in self.blocks: x = block(x)     # same shape, n_blocks times
        return self.head(x)                        # -> (batch, N_BASES, 4)

**`TODO-1`.** Build a `Basecaller` with sizes you choose, and
train it on the *same* data as the CNN. Compare the two accuracy columns with what the CNN
managed. Note the parameter counts that `train` prints.
Comparing two architectures is only fair if they are close in size, so check yours
against the CNN's.

In [ ]:
# TODO-1: pick your four sizes and train. There are no defaults on `Basecaller` -
#         leaving a size out is a TypeError, because choosing them is the exercise.
#
# Every knob `train` takes. All of them except the model have a default, so pass only the
# ones you want to change:
#
#     train(model, "a label for the printed line",
#           steps=...,     # how many optimiser steps
#           batch=...,     # reads per step
#           lr=...,        # learning rate
#           seed=...,      # seeds the batch order, not the weights - to fix the weights
#                          #   too, call torch.manual_seed(...) before you build
#           every=...,     # how often to evaluate and print
#           noise=...,     # forwarded to the generator
#           uneven=...)    # forwarded to the generator
#
# It hands back a dict: r["acc"] / r["fwd"] / r["rev"], r["params"], and r["hist"].

model = Basecaller(d_model=..., n_heads=..., n_blocks=..., d_ff=...)
tx_run = train(model, "Transformer")

## 4. Harder data, bigger models (TODO-2)

Vary the model sizes and the two data settings, and keep the runs you need to justify a
choice of configuration. `train` hands back `hist`, so you can plot learning curves as well as
final numbers.

You do not need long runs to compare sizes. The gap between a small and a large model is
clear after about `steps=1000`, and training longer mostly lifts both together. Sweep short,
then spend one long run on the configuration you settle on.

In [ ]:
# TODO-2: design the comparison yourself, then run it here. Section 3 shows how to
#         call `train`, with every argument it takes written out.
#
#         `train` returns a dict per run: r["acc"] / r["fwd"] / r["rev"] are the final
#         numbers, r["params"] the size, and r["hist"] the progress points, each with
#         "step", "loss", "acc", "fwd", "rev" - which is what you plot a curve from.
#
#         Keep every run you make, including the ones that turn out badly. The report
#         asks for the evidence behind your choice, not only the choice.

### Save the model you want graded

Your performance grade is computed from this file, not from a number in your report. Save the
model you settled on, then load it straight back and score it - if the file is wrong you want
to find that out here rather than after you submit.

On Colab the file lands in the session and disappears when the session ends, so download it
from the Files pane before you close the tab.

In [ ]:
#@title Save and check
BEST = ...        # <-- the model you want graded

torch.save({"sizes": BEST.sizes, "state": BEST.state_dict()}, "basecaller.pt")

ck = torch.load("basecaller.pt", weights_only=True)
reloaded = Basecaller(**ck["sizes"]).to(DEVICE)
reloaded.load_state_dict(ck["state"])
acc, fwd, rev = evaluate(reloaded)
print(f"saved basecaller.pt   {ck['sizes']}")
print(f"reloaded and scored   acc {acc:.3f}  (forward {fwd:.3f} | reverse {rev:.3f})")

## 5. Ablation: turn positional encoding off (TODO-3)

Rebuild with `use_pe=False`, keeping every other size fixed, and retrain.

In [ ]:
# TODO-3: rebuild with use_pe=False and compare. Keep every other size identical to the
#         model you trained in Section 3 - change one thing at a time, or you are
#         comparing two differences at once.
m = Basecaller(d_model=..., n_heads=..., n_blocks=..., d_ff=..., use_pe=False)
no_pe_run = train(m, "no-PE")

## 6. What did it learn?

For one read we can plot, for every position the model computes, how much weight it put on
every position it could have looked at. That is one square grid per attention head. Row $i$ is the
position being computed, column $j$ is the position it looked at, and brightness is the
weight. Every row sums to 1, so an even split over 60 positions would be 0.017 everywhere,
and a bright cell means that one position got a large share of the attention.

Three shapes to know by sight, drawn below:

- a bright **main diagonal**: each position attending to itself and its neighbours
- a bright **vertical stripe**: every position attending to the same single column
- a bright **anti-diagonal**: position $i$ attending to position $59-i$

Which attention head does which moves around between training runs, and some do none of them.

In [ ]:
#@title The three shapes, drawn by hand { display-mode: "form" }
# Not from a model. These are the patterns worth recognising in your own grids below.
n = 24
q = np.arange(n)
shapes = {
    "local\neach position looks at\nitself and its neighbours":
        np.exp(-((q[:, None] - q[None, :]) ** 2) / 2.0),
    "one fixed place\nevery position looks\nat the same spot":
        np.tile((q == 0).astype(float), (n, 1)),
    "the mirror\neach position looks at\nthe far end of the read":
        (q[:, None] == (n - 1 - q)[None, :]).astype(float),
}
fig, axes = plt.subplots(1, 3, figsize=(9, 3.4))
for a, (title, M) in zip(axes, shapes.items()):
    a.imshow(M, cmap="magma", vmin=0, vmax=1)
    a.set_title(title, fontsize=8)
    a.set_xlabel("position looked at", fontsize=8)
    a.set_xticks([0, n - 1]); a.set_yticks([0, n - 1]); a.tick_params(labelsize=7)
axes[0].set_ylabel("position being computed", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
#@title Attention grids, one panel per attention head { display-mode: "form" }
# uses the `model` you trained in Section 3
while True:
    sig, lab, fwd = make_batch(1)
    if fwd[0] == 0: break

model.eval()
with torch.no_grad():
    x = model.tokenizer(sig.to(DEVICE))
    x = model.pos(x) if model.pos is not None else x
    grids = []
    for blk in model.blocks:
        _, att = blk.attn(blk.ln1(x), return_attn=True)
        grids.append(att[0].cpu().numpy())        # (n_heads, N_BASES, N_BASES)
        x = blk(x)

n_blocks, n_heads = len(grids), grids[0].shape[0]
fig, axes = plt.subplots(n_blocks, n_heads, squeeze=False,
                         figsize=(2.2 * n_heads, 2.3 * n_blocks))
ticks = [0, N_BASES // 2, N_BASES - 1]
for b in range(n_blocks):
    for h in range(n_heads):
        a = axes[b][h]
        # scale each panel to its own peak, or a diffuse head renders as solid black
        a.imshow(grids[b][h], cmap="magma", vmin=0, vmax=max(float(grids[b][h].max()), .15))
        a.set_title(f"block {b}, head {h}", fontsize=8)
        a.set_xticks(ticks); a.set_yticks(ticks); a.tick_params(labelsize=6)
        if h == 0: a.set_ylabel("position computed", fontsize=7)
        if b == n_blocks - 1: a.set_xlabel("position looked at", fontsize=7)
plt.tight_layout(); plt.show()

### Turn a head off (TODO-4)

The heads are computed side by side and joined before the block's output layer, so one head
can be silenced by zeroing its slice of that join. Everything else - the other heads, the
residual connections, the later blocks - is untouched.

In [ ]:
#@title kill_head
def kill_head(model, block, head):
    """Silence one head. Returns a handle; call .remove() on it to put the head back."""
    attn = model.blocks[block].attn
    def hook(module, args):
        y = args[0].clone()
        y[..., head * attn.dk:(head + 1) * attn.dk] = 0
        return (y,)
    return attn.out.register_forward_pre_hook(hook)

In [ ]:
# TODO-4: silence each head in turn and record what the model loses. `evaluate` returns
#         (overall, forward, reverse), so compare against the untouched model and keep the
#         three numbers per head. Put the handle back with .remove() before the next one,
#         or you will be measuring several dead heads at once.
base = evaluate(model)
print("baseline", base)

for b in range(len(model.blocks)):
    for h in range(model.blocks[b].attn.h):
        ...